In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

catalog = "fortum_challenge_data"
results_schema = "04_results"
pred_table = "consumption_hourly_predictions_weather_null"

# Read prediction table
df_pred = spark.table(f"{catalog}.{results_schema}.{pred_table}")

# Keep only needed columns
df_pred = df_pred.select(
    F.col("timestamp_utc"),
    F.col("group_id"),
    F.col("target_consumption")
)

# Get group_ids in ascending order
group_ids = [
    row["group_id"]
    for row in df_pred.select("group_id").distinct().orderBy("group_id").collect()
]

# Pivot from long to wide
df_deliverable = (
    df_pred
    .groupBy("timestamp_utc")
    .pivot("group_id", group_ids)
    .agg(F.first("target_consumption"))
    .orderBy("timestamp_utc")
)

# Rename timestamp column if needed
df_deliverable = df_deliverable.withColumnRenamed("timestamp_utc", "measured_at")

deliverable_schema = "05_deliverables"
deliverable_table = "consumption_deliverables_hourly"

(df_deliverable
    .write
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{deliverable_schema}.{deliverable_table}")
)

display(df_deliverable)